# Contingency tables

## 1) Import stuff  

In [1]:
import geopandas
import regionmask
import os
import xarray as xr
import pandas as pd
import numpy as np
from shapely.geometry import box

## 2) Change these
* Input parameters to change the region between a lat-lon box or Longhurst Province
    - ProvCode = "CCAL"; our any other Longhurt Province 
    - ProvCode = 'box'; defined box, set with lat_rgn = [30, 48] and lon_rgn = [-130, -115]
* Comparison data
    - start and end dates; startyear=YYYYMMDD and endyear=YYYYMMDD
    - data source
    - prcnt_keep; between 0 and 1, trends can be calculated on time series with missing values
    - nvvar; variable that the trend has been calculated
    - timeseries_type; either 'data' or 'anom'

In [2]:
# a) 
ProvCode = "CCAL"
# ProvCode = "box"
lat_rgn = [30, 48]
lon_rgn = [-130, -115]

# b) comparison data
startyear = 201301
endyear = 202212
source1 = 'nasa'
source2 = 'modis'
prcnt_keep = 0.5
ncvar = 'productivity'
# timeseries_type = 'data'
timeseries_type = 'anom'

# c)
alpha = 0.05


# d) set directories to link to the trend netcdf files
BASE_DIR = '/home/isaac/data_files/Work/CW/monthly/'
BIN_DIR = os.path.join(BASE_DIR, 'bin2')
DATA_DIR = os.path.join(BASE_DIR, 'data', 'monthly')
WORK_DIR = os.path.join(BASE_DIR, 'data', 'work')
RESOURCES_DIR = os.path.join(BASE_DIR, 'resources')

ifile_tmpl = '{}_{}_trend_month_{}_{}_{}_{}_{:03d}percent.nc'

## 3) Region  
* Load Longhurst Provinces

In [3]:
if ProvCode == 'box':
    rgn_wnt = [lon_rgn, lat_rgn]
else:
    # a) Load the Longhurst Provinces shape files into a geopandas dataframe
    shape_path = os.path.join('/home/isaac/data_files/Work/CW/primprod/dataset_comparison/',
                              'resources',
                              'longhurst_v4_2010',
                              'Longhurst_world_v4_2010.shp')
    shapefiles = geopandas.read_file(shape_path)
    shapefiles.head(8)

    # b) Locate the row with the ProvCode code
    prov_wnt = shapefiles.loc[shapefiles["ProvCode"] == ProvCode]

    # c) Find the coordinates of the bounding box
    #    The bounding box is the smallest rectangle that will completely enclose the province.
    #    We will use the bounding box coordinates to subset the satellite data
    gs_bnds = prov_wnt.bounds

    # d)
    rgn_wnt = [[gs_bnds.minx.item(), gs_bnds.maxx.item()],
               [gs_bnds.miny.item(), gs_bnds.maxy.item()]]

DataSourceError: /home/isaac/data_files/Work/CW/primprod/dataset_comparison/resources/longhurst_v4_2010/Longhurst_world_v4_2010.shp: No such file or directory

## 4) Load the data
    - Load the trend data for the two source data sets.
    - Save the lat and lon coordinates, will use these to get the exact positions of the corners for the "box" region.
    - Coordinates are used for the "box" shape file.

In [ ]:
source_list = [source1, source2]
ds1_list = []
for source in source_list:
    idir = os.path.join(DATA_DIR,
                        'trends',
                        source,
                        )

    ifile = ifile_tmpl.format(ncvar, timeseries_type, source, '9km', startyear, endyear, int(prcnt_keep*100))

    ds1 = xr.open_dataset(os.path.join(idir, ifile))
    # ds1_list.append(ds1.sel(latitude=slice(lat_rgn[0], lat_rgn[1])).sel(longitude=slice(lon_rgn[0], lon_rgn[1])))
    ds1_list.append(ds1)
lon1t = ds1['longitude'].sel(longitude=slice(rgn_wnt[0][0], rgn_wnt[0][1])).data
lat1t = ds1['latitude'].sel(latitude=slice(rgn_wnt[1][1], rgn_wnt[1][0])).data


## 5) Get region from the shape file
    - ProvCode='box', use geopandas to create a new GeoDataFrame 
    - Provcode='Longhurt', region from the province shape file

In [ ]:
# Create the region from the shape file
if ProvCode == 'box':
    box_rgn = box(lon1t[0],lat1t[0],lon1t[-1],lat1t[-1])
    newdata = geopandas.GeoDataFrame(index=[0], crs='epsg:4326', geometry=[box_rgn])
    region = regionmask.from_geopandas(newdata)
else:
    region = regionmask.from_geopandas(prov_wnt)

## 6) Mask the data based on the region
    - Mask the trend data for both source types
    - Apply the mask to the DataArray using the 'where' function.
    - The 'where' function sets any gridpoints outside the mask to a NaN value.

In [ ]:
da_beta_list = []
da_pval_list = []
for i in range(len(source_list)):
    da_beta = ds1_list[i]['beta'].squeeze().sel(latitude=lat1t).sel(longitude=lon1t)
    da_pval = ds1_list[i]['pval'].squeeze().sel(latitude=lat1t).sel(longitude=lon1t)

    #   Create the mask
    mask = region.mask(da_beta.longitude, da_beta.latitude)

    # Apply mask the the satellite data
    da_beta_mask = da_beta.where(mask == region.numbers[0])
    da_beta_list.append(da_beta_mask)
    da_pval_mask = da_pval.where(mask == region.numbers[0])
    da_pval_list.append(da_pval_mask)


## 7) For the two data types find grid locations that have trend values
    - Find common grid points that have trend values.
    - Trend data is a 2D matrix shaped [ny, nx]
    - Work with vectors instead of 2D matrix
    - Vectors allow us to use numpy in1d function to find indices between the two data types that have trend values.
    - The two data sets can have different missing values, due to different land mask and other factors.
    - The steps are: reshape from 2d to 1d (using np.reshape), find index missing values for each data set (using np.isfinite), find common indices between the two datasets (using np.in1d)
    - This code block will create beta and pval vectors used for the contigency tables.
    - It will set the varialbe 'num_common' that is used as the denominator for the percentages given in the tables


In [ ]:
# get the common 
beta_re_list = []
pval_re_list = []
ind_re_list = []
for i in range(len(source_list)):
    ny, nx = da_beta_list[i].shape
    beta_re = np.reshape(da_beta_list[i].data, ny*nx)
    pval_re = np.reshape(da_pval_list[i].data, ny*nx)
    ind_re = np.isfinite(beta_re).nonzero()[0]

    # check for unusual case of beta == 0, make it a slightly larger 0
    in0 = np.where(beta_re == 0.0)
    beta_re[in0] = 0.00001
    pval_re[in0] = 0.00001

    beta_re_list.append(beta_re)
    pval_re_list.append(pval_re)
    ind_re_list.append(ind_re)

ia12 = np.in1d(ind_re_list[0], ind_re_list[1])
ia21 = np.in1d(ind_re_list[1], ind_re_list[0])
beta_common = [beta_re_list[0][ind_re_list[0]][ia12], beta_re_list[1][ind_re_list[1]][ia21]]
pval_common = [pval_re_list[0][ind_re_list[0]][ia12], pval_re_list[1][ind_re_list[1]][ia21]]

num_common = len(beta_common[0])

## 8) Create the $2x2$ contigency table
    - The table describes the trends, significance of the trends is not considered.
    - Create column & row labels. 

In [ ]:
operator_lbl = [r'$\beta>=0$', r'$\beta<0$']
clmn22_lbl = []
row22_lbl = []
for i in range(2):
    clmn22_lbl.append('{} {}'.format(source_list[0].upper(), operator_lbl[i]))
    row22_lbl.append('{} {}'.format(source_list[1].upper(), operator_lbl[i]))


In [ ]:
pn_list = [[1, 1], [-1, 1], [1, -1], [-1, -1]]
table22_vec = np.zeros(len(pn_list))
for i in range(len(pn_list)):
    pn01 = pn_list[i]
    in0 = np.where(pn01[0]*beta_common[0] > 0)[0]
    in1 = np.where(pn01[1]*beta_common[1] > 0)[0]
    ia = np.in1d(in0, in1)
    num1 = len(ia.nonzero()[0])
    table22_vec[i] = 100*num1/num_common

table22_mtrx = table22_vec.reshape(2, 2)


In [ ]:
# Create a Pandas DataFrame from the 2x2 table
df22 = pd.DataFrame(table22_mtrx, columns=clmn22_lbl)
df22 = df22.set_index(np.array(row22_lbl))


## 9) Create $3 x 3$ contigency table
    - The 3x3 table describes the trends and their significance.
    - Initialize the table and create column & row labels.
    - The lower-right 2 rows and columns are calculated the same as the 2x2 matrix but with common points that are significant (< alpha).
    - The first rows and columns describe when the common points are either both non-signicant (> alpha) or only one of the datasets is non-signicicant.
    - The cells of the first row and column are calculate individually (probably can be done with a loop)

In [ ]:
table33_mtrx = np.zeros([3, 3])

operator_lbl = ['n.s.', r'$\beta>=0$', r'$\beta<0$']
clmn33_lbl = []
row33_lbl = []
for i in range(3):
    clmn33_lbl.append('{} {}'.format(source_list[0].upper(), operator_lbl[i]))
    row33_lbl.append('{} {}'.format(source_list[1].upper(), operator_lbl[i]))

    - Calculate the first row and column.
    - Calculate the 5 cells individually. 

In [ ]:
# both ns (ns=not significant)
in1ns = np.where(pval_common[0] >= alpha)[0]
in2ns = np.where(pval_common[1] >= alpha)[0]
ia_ns = np.in1d(in1ns, in2ns).nonzero()[0]
table33_mtrx[0, 0] = 100*len(ia_ns)/num_common

# beta1 ns beta2 pos and neg
in2sig = np.where(pval_common[1][in1ns] < alpha)[0]
beta2_sig = beta_common[1][in1ns][in2sig]
in2_sig_pos = np.where(beta2_sig >= 0)[0]
in2_sig_neg = np.where(beta2_sig < 0)[0]
table33_mtrx[1, 0] = 100*len(in2_sig_pos)/num_common
table33_mtrx[2, 0] = 100*len(in2_sig_neg)/num_common

# beta2 ns beta1 pos and neg
in1sig = np.where(pval_common[0][in2ns] < alpha)[0]
beta1_sig = beta_common[0][in2ns][in1sig]
in1_sig_pos = np.where(beta1_sig >= 0)[0]
in1_sig_neg = np.where(beta1_sig < 0)[0]
table33_mtrx[0, 1] = 100*len(in1_sig_pos)/num_common
table33_mtrx[0, 2] = 100*len(in1_sig_neg)/num_common

    - Calculate the lower-right 4 cells.
    - Use only common grid cells that both have significant trends (<alptha).
    - These are calculated the same as the 2x2.

In [ ]:
# both sig
in1sig = np.where(pval_common[0] < alpha)[0]
in2sig = np.where(pval_common[1] < alpha)[0]
ia = np.in1d(in1sig, in2sig).nonzero()[0]
ib = np.in1d(in2sig, in1sig).nonzero()[0]
data_re_list = [beta_common[0][in1sig][ia], beta_common[1][in2sig][ib]]

pn_list = [[1, 1], [-1, 1], [1, -1], [-1, -1]]
table33_vec = np.zeros(len(pn_list))
for i in range(len(pn_list)):
    pn01 = pn_list[i]
    in0 = np.where(pn01[0]*data_re_list[0] > 0)[0]
    in1 = np.where(pn01[1]*data_re_list[1] > 0)[0]
    ia = np.in1d(in0, in1)
    num1 = len(ia.nonzero()[0])
    table33_vec[i] = 100*num1/num_common

table33_mtrx[1:, 1:] = np.reshape(table33_vec, [2, 2])

    - Create a Pandas DataFrame from the 3x3 table.

In [ ]:
df33 = pd.DataFrame(table33_mtrx, columns=clmn33_lbl)
df33 = df33.set_index(np.array(row33_lbl))
df33 = np.round(df33*100)/100

## 10) Calculate $\kappa$ for the $2 x 2$ and $3 x 3 $ tables

In [ ]:
# kappa for 2 x 2

row_sum = np.sum(table22_mtrx, axis=1)
clmn_sum = np.sum(table22_mtrx, axis=0)
trc_sum = np.trace(table22_mtrx)
overall_sum = np.sum(row_sum)

ef1 = row_sum*clmn_sum/overall_sum
ef1_sum = np.sum(ef1)

k22 = (trc_sum - ef1_sum)/(overall_sum - ef1_sum)


In [ ]:
# kappa for 3 x 3

row_sum = np.sum(table33_mtrx, axis=1)
clmn_sum = np.sum(table33_mtrx, axis=0)
trc_sum = np.trace(table33_mtrx)
overall_sum = np.sum(row_sum)

ef1 = row_sum*clmn_sum/overall_sum
ef1_sum = np.sum(ef1)

k33 = (trc_sum - ef1_sum)/(overall_sum - ef1_sum)


## 11) Display tables and kappa

In [ ]:
print('Province: {}\n'.format(ProvCode))
print('Region: {:6.2f} to {:6.2f}, {:6.2f} to {:6.2f}\n'.format(rgn_wnt[0][0], rgn_wnt[0][1], rgn_wnt[1][0], rgn_wnt[1][1]))
print('Contigency table 2 x 2:\n')
print(df22)
print('\n2 x 2 kappa: {:5.2f}\n'.format(k22))
print('Contigency table 3 x 3:\n')
print(df33)
print('\n3 x 3 kappa: {:5.2f}\n'.format(k33))
